In [ ]:
"""
finetune_lora_tictactoe.py
==========================
Fine-tuning de Qwen3-14B-Instruct con LoRA para Tic-tac-toe / Suicide.

Hardware objetivo : 1× NVIDIA A100 40 GB
Modelo            : Qwen/Qwen3-14B-Instruct  (~28 GB en bfloat16)
Dataset           : JSONL en formato Alpaca (instruction / input / output)
Tamaño dataset    : ~1 500 jugadas

Formato Alpaca esperado por ejemplo:
  {
    "instruction": "You are an expert Tic-Tac-Toe strategist...",
    "input":       "GDL Board State: [...]\nLegal Moves: [...]\nAction for Player: X",
    "output":      "Analysis: ...\nMove: ['mark', '1', '1']"
  }

El prompt se construye con la plantilla Alpaca estándar:
  ### Instruction:
  {instruction}

  ### Input:
  {input}

  ### Response:
  {output}

La loss solo se calcula sobre los tokens de "### Response:" en adelante.

Dependencias:
  pip install transformers peft trl datasets accelerate
"""

# =============================================================================
# 1. IMPORTS
# =============================================================================

import os
import json
import logging
from dataclasses import dataclass, field
from typing import Literal, Optional

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM, SFTConfig

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Plantilla Alpaca estándar
ALPACA_TEMPLATE = (
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n{output}"
)

# Token que marca el inicio de la respuesta; el collator calcula loss solo desde aquí.
RESPONSE_TEMPLATE = "### Response:\n"


# =============================================================================
# 2. CONFIGURACIÓN PRINCIPAL
# =============================================================================

@dataclass
class LoRAConfig:
    r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.10
    target_modules: list = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ])
    bias: str = "none"
    init_lora_weights: bool = True


@dataclass
class ModelConfig:
    model_name: str = "Qwen/Qwen3-14B"
    max_seq_length: int = 512
    dtype: torch.dtype = torch.bfloat16
    trust_remote_code: bool = True
    token: Optional[str] = os.getenv("HF_TOKEN")
    attn_implementation: str = "sdpa"

@dataclass
class TrainConfig:
    output_dir: str = "./checkpoints"
    num_train_epochs: int = 5
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 8

    gradient_checkpointing: bool = True
    learning_rate: float = 5e-5
    warmup_ratio: float = 0.10
    weight_decay: float = 0.01

    lr_scheduler_type: str = "cosine"
    optim: str = "paged_adamw_32bit"
    fp16: bool = False
    bf16: bool = True
    max_grad_norm: float = 1.0

    # Logging
    logging_steps: int = 5
    report_to: str = "none"

    # Guardado de checkpoints
    save_steps: int = 50
    save_total_limit: int = 3

    # Eficiencia del entrenamiento
    group_by_length: bool = True
    dataloader_num_workers: int = 2
    seed: int = 42

    #Parámetro para definir el ID del repositorio en Hugging Face
    push_to_hub_id: Optional[str] = None

@dataclass
class DataConfig:
    dataset_path: str = "./Expert_Dataset_TicTacToe_1500.jsonl"


# =============================================================================
# 3. CARGA Y PREPROCESAMIENTO DEL DATASET
# =============================================================================

def load_and_format_dataset(data_cfg: DataConfig) -> dict:
    """
    Carga el JSONL y aplica la plantilla Alpaca estándar a cada ejemplo.

    Transformación por ejemplo:
      instruction + input + output  →  texto único con plantilla Alpaca

    La plantilla queda:
      ### Instruction:
      You are an expert Tic-Tac-Toe strategist...

      ### Input:
      GDL Board State: [...]
      Legal Moves: [...]
      Action for Player: X

      ### Response:
      Analysis: ...
      Move: ['mark', '1', '1']

    El DataCollatorForCompletionOnlyLM usará RESPONSE_TEMPLATE = "### Response:\n"
    para enmascarar todo lo anterior y calcular la loss solo sobre la respuesta.
    """
    logger.info(f"Cargando dataset desde: {data_cfg.dataset_path}")
    raw = load_dataset("json", data_files=data_cfg.dataset_path, split="train")
    logger.info(f"Total de ejemplos cargados: {len(raw)}")

    def apply_alpaca_template(example: dict) -> dict:
        text = ALPACA_TEMPLATE.format(
            instruction = example["instruction"],
            input       = example["input"],
            output      = example["output"],
        )
        return {"text": text}

    formatted = raw.map(apply_alpaca_template, remove_columns=raw.column_names)
    logger.info(f"Total de ejemplos para entrenamiento: {len(formatted)}")
    return formatted


# =============================================================================
# 4. PIPELINE PRINCIPAL
# =============================================================================

def load_model_and_tokenizer(model_cfg: ModelConfig, train_cfg: TrainConfig):
    logger.info(f"Cargando modelo: {model_cfg.model_name}")

    model = AutoModelForCausalLM.from_pretrained(
        model_cfg.model_name,
        torch_dtype           = model_cfg.dtype,
        device_map            = "auto",
        attn_implementation   = model_cfg.attn_implementation,
        trust_remote_code     = model_cfg.trust_remote_code,
        token                 = model_cfg.token,
    )

    if train_cfg.gradient_checkpointing:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )

    tokenizer = AutoTokenizer.from_pretrained(
        model_cfg.model_name,
        trust_remote_code = model_cfg.trust_remote_code,
        token             = model_cfg.token,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

    return model, tokenizer


def attach_lora(model, lora_cfg: LoRAConfig):
    config = LoraConfig(
        r                 = lora_cfg.r,
        lora_alpha        = lora_cfg.lora_alpha,
        lora_dropout      = lora_cfg.lora_dropout,
        target_modules    = lora_cfg.target_modules,
        bias              = lora_cfg.bias,
        task_type         = TaskType.CAUSAL_LM,
        init_lora_weights = lora_cfg.init_lora_weights,
    )
    model = get_peft_model(model, config)
    model.print_trainable_parameters()
    return model


def run_training(
    model_cfg : ModelConfig,
    lora_cfg  : LoRAConfig,
    train_cfg : TrainConfig,
    data_cfg  : DataConfig,
):
    # ── 4.1 Modelo ───────────────────────────────────────────────────────────
    model, tokenizer = load_model_and_tokenizer(model_cfg, train_cfg)
    model = attach_lora(model, lora_cfg)

    # ── 4.2 Dataset ──────────────────────────────────────────────────────────
    dataset = load_and_format_dataset(data_cfg)

    # ── 4.3 Data collator ────────────────────────────────────────────────────
    # Enmascara instruction + input; la loss solo se calcula sobre el output.
    collator = DataCollatorForCompletionOnlyLM(
        response_template = RESPONSE_TEMPLATE,
        tokenizer         = tokenizer,
    )

    # ── 4.4 SFTConfig ────────────────────────────────────────────────
    output_dir = os.path.join(train_cfg.output_dir, f"Qwen3-14B_expert")

    training_args = SFTConfig(
        # General
        output_dir                  = output_dir,
        seed                        = train_cfg.seed,

        # Épocas y batches
        num_train_epochs            = train_cfg.num_train_epochs,
        per_device_train_batch_size = train_cfg.per_device_train_batch_size,
        per_device_eval_batch_size  = train_cfg.per_device_train_batch_size,
        gradient_accumulation_steps = train_cfg.gradient_accumulation_steps,
        group_by_length             = train_cfg.group_by_length,
        dataloader_num_workers      = train_cfg.dataloader_num_workers,

        # Memoria
        gradient_checkpointing = train_cfg.gradient_checkpointing,

        # Optimizador y learning rate
        learning_rate     = train_cfg.learning_rate,
        warmup_ratio      = train_cfg.warmup_ratio,
        weight_decay      = train_cfg.weight_decay,
        lr_scheduler_type = train_cfg.lr_scheduler_type,
        optim             = train_cfg.optim,
        max_grad_norm     = train_cfg.max_grad_norm,

        # Precisión
        fp16 = train_cfg.fp16,
        bf16 = train_cfg.bf16,

        # Logging
        logging_steps  = train_cfg.logging_steps,
        report_to      = train_cfg.report_to,

        # Guardado de checkpoints
        save_steps       = train_cfg.save_steps,
        save_total_limit = train_cfg.save_total_limit,

        dataset_text_field          = "text",
        max_seq_length              = model_cfg.max_seq_length,
        dataset_num_proc            = 2,
        packing                     = False,

    )

    # ── 4.5 SFTTrainer ───────────────────────────────────────────────────────
    trainer = SFTTrainer(
        model              = model,
        processing_class   = tokenizer,
        train_dataset      = dataset,
        args               = training_args,
        data_collator      = collator,
    )

    # ── 4.6 Entrenamiento ────────────────────────────────────────────────────
    logger.info("Iniciando entrenamiento...")
    trainer.train()

    # ── 4.7 Guardar adaptador ────────────────────────────────────────────────
    final_dir = os.path.join(output_dir, "final_adapter")
    trainer.model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)

    with open(os.path.join(final_dir, "experiment_config.json"), "w") as f:
        json.dump({
            "model":    {**model_cfg.__dict__, "dtype": str(model_cfg.dtype)},
            "lora":     {**lora_cfg.__dict__},
            "training": {**train_cfg.__dict__},
            "data":     {**data_cfg.__dict__},
        }, f, indent=2)

    logger.info(f"Adaptador guardado en: {final_dir}")

    # ── 4.8 Subir a Hugging Face (Opcional) ───────────────────────────
    if train_cfg.push_to_hub_id:
        logger.info(f"Subiendo adaptador a Hugging Face Hub: {train_cfg.push_to_hub_id}...")
        try:
            trainer.model.push_to_hub(train_cfg.push_to_hub_id, token=model_cfg.token)
            tokenizer.push_to_hub(train_cfg.push_to_hub_id, token=model_cfg.token)
            logger.info("✓ ¡Adaptador subido con éxito al Hub!")
        except Exception as e:
            logger.error(f"Error al subir el modelo a Hugging Face: {e}")

    return final_dir


# =============================================================================
# 5. ENTRYPOINT CLI
# =============================================================================

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Fine-tuning LoRA de Qwen3-14B para Tic-Tac-Toe / Suicide"
    )
    parser.add_argument("--dataset", type=str,   default="./Expert_Dataset_TicTacToe_1500.jsonl")
    parser.add_argument("--epochs",  type=int,   default=5)
    parser.add_argument("--lr",      type=float, default=5e-5)
    parser.add_argument("--lora_r",  type=int,   default=16)
    # Argumento para recibir el repositorio destino
    parser.add_argument("--push_to_hub_id", type=str, default=None,
                        help="ID del repositorio en Hugging Face, HailNicol/qwen3-14b-ttt-ftl-v4")
    args = parser.parse_args()

    saved_to = run_training(
        model_cfg = ModelConfig(),
        lora_cfg  = LoRAConfig(r=args.lora_r, lora_alpha=args.lora_r * 2),
        train_cfg = TrainConfig(
            num_train_epochs=args.epochs,
            learning_rate=args.lr,
            push_to_hub_id=args.push_to_hub_id
        ),
        data_cfg  = DataConfig(dataset_path=args.dataset),
    )
    print(f"\n✓ Proceso finalizado.")